# LASSER on Kaggle

GEARS with an optional GSR-style learned co-expression graph. Needs **Internet on** (GEARS downloads Norman and the GO graph from Harvard Dataverse) and a **GPU (T4)**.

Cells: install, get the code, run the tests (before any real training), baseline run, graph-learning run, results.

In [1]:
# 1. Dependencies. GEARS is vendored inside lasser/ (lasser/gears), so no cell-gears install.
#    Kaggle already has torch, numpy, pandas, scipy, scikit-learn, networkx, tqdm, requests.
!pip install -q torch_geometric scanpy pytest

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 25.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 190.2/190.2 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 376.9/376.9 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 230.5/230.5 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 2.3 MB/s eta 0:00:00


In [2]:
# 2. Get the code: clone, then move the self-contained package next to the notebook.
REPO_URL = "https://github.com/LegendaryDarkKnight/LASSER"   # <- set this
REPO_DIR = "/kaggle/working/lasser-repo"
!rm -rf {REPO_DIR} /kaggle/working/lasser
!git clone -q {REPO_URL} {REPO_DIR}
!mv {REPO_DIR}/lasser /kaggle/working/lasser
!git -C {REPO_DIR} log -1 --oneline

02a13c9 (HEAD -> main, origin/main, origin/HEAD) fix: format issue fixing


In [3]:
# 3. Tests 1-8 (tiny subsets; first run also downloads Norman, ~10-20 min in total).
%env LASSER_DATA_DIR=/kaggle/working/data
%env LASSER_TEST_DIR=/kaggle/working/lasser_tests
!cd /kaggle/working && python -m pytest {REPO_DIR}/tests -m "not smoke" -p no:cacheprovider -rA -s --tb=short 2>&1 | tee /kaggle/working/test_output.txt | grep -vE "it/s\]|s/it\]"

env: LASSER_DATA_DIR=/kaggle/working/data
env: LASSER_TEST_DIR=/kaggle/working/lasser_tests
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0
rootdir: /kaggle/working
plugins: fast-array-utils-1.5.1, zarr-3.4.0, typeguard-4.6.0, anyio-4.14.2, langsmith-0.12.1
collected 16 items

lasser-repo/tests/test_00_packaging.py ..lasser modules loaded: ['lasser', 'lasser._eval', 'lasser.config', 'lasser.evaluation', 'lasser.gears', 'lasser.gears.data_utils', 'lasser.gears.gears', 'lasser.gears.inference', 'lasser.gears.model', 'lasser.gears.pertdata', 'lasser.gears.utils', 'lasser.gears.version', 'lasser.run', 'lasser.tracking', 'lasser.utils']
.
lasser-repo/tests/test_01_parity.py Found local copy...
Found local copy...
Found local copy...
These perturbations are not in the GO graph and their perturbation can thus not be predicted
['RHOXF2BB+ctrl' 'LYL1+IER5L' 'ctrl+IER5L' 'KIAA1804+ctrl' 'IER5L+ctrl'
 'R

In [4]:
# 3b. Optional test 9: full Norman, 1 epoch, flag off and on, checks the T4 memory budget.
%env LASSER_SMOKE=1
!cd /kaggle/working && python -m pytest {REPO_DIR}/tests/test_09_smoke.py -p no:cacheprovider -rA -s --tb=short 2>&1 | tee /kaggle/working/smoke_output.txt | grep -vE "it/s\]|s/it\]"
%env LASSER_SMOKE=0

env: LASSER_SMOKE=1
============================= test session starts ==============================
platform linux -- Python 3.13.15, pytest-8.4.2, pluggy-1.6.0
rootdir: /kaggle/working
plugins: fast-array-utils-1.5.1, zarr-3.4.0, typeguard-4.6.0, anyio-4.14.2, langsmith-0.12.1
collected 2 items

lasser-repo/tests/test_09_smoke.py Found local copy...
Found local copy...
Found local copy...
These perturbations are not in the GO graph and their perturbation can thus not be predicted
['RHOXF2BB+ctrl' 'LYL1+IER5L' 'ctrl+IER5L' 'KIAA1804+ctrl' 'IER5L+ctrl'
 'RHOXF2BB+ZBTB25' 'RHOXF2BB+SET']
Local copy of pyg dataset is detected. Loading...
Done!
2026-10-06 15:23:31,023 INFO lasser.tracking: run 84c27427 -> /kaggle/working/lasser_tests/runs/norman_seed1_base_20261006-152331_smoke
2026-10-06 15:23:31,091 INFO lasser.run: device: cuda:0 (Tesla T4, 14.6 GB) (GEARS training)
Local copy of split is detected. Loading...
Simulation split test composition:
combo_seen0:9
combo_seen1:43
combo_seen2:1

In [5]:
# 4. Load the data once and reuse it for both runs.
from lasser import GSRConfig, LasserConfig, load_pert_data, load_run, run_experiment

common = dict(dataset="norman", seed=1, epochs=20,
              data_dir="/kaggle/working/data", out_dir="/kaggle/working/runs",
              cache_dir="/kaggle/working/cache", repo_dir=REPO_DIR)
pert_data = load_pert_data(LasserConfig(**common))

Found local copy...
Found local copy...
Found local copy...
These perturbations are not in the GO graph and their perturbation can thus not be predicted
['RHOXF2BB+ctrl' 'LYL1+IER5L' 'ctrl+IER5L' 'KIAA1804+ctrl' 'IER5L+ctrl'
 'RHOXF2BB+ZBTB25' 'RHOXF2BB+SET']
Local copy of pyg dataset is detected. Loading...
Done!


In [6]:
# 5. Baseline: graph_learning=False is plain GEARS (plus logging and saving).
base = run_experiment(LasserConfig(**common, graph_learning=False), pert_data)
base["run_dir"]

2026-10-06 15:42:35,532 INFO lasser.tracking: run f71d4a24 -> /kaggle/working/runs/norman_seed1_base_20261006-154235
2026-10-06 15:42:35,592 INFO lasser.run: device: cuda:0 (Tesla T4, 14.6 GB) (GEARS training)


Local copy of split is detected. Loading...
Simulation split test composition:
combo_seen0:9
combo_seen1:43
combo_seen2:19
unseen_single:36
Done!
Creating dataloaders....
Done!


here1
2026-10-06 15:42:35,608 INFO lasser.tracking: split hash f93e7d59ee239960 (copied ['norman_simulation_1_0.75_subgroup.pkl', 'norman_simulation_1_0.75.pkl'])
2026-10-06 15:42:35,610 INFO lasser.utils: [gears_init] start


Found local copy...


2026-10-06 15:42:55,107 INFO lasser.utils: [gears_init] done in 19.5s, peak CUDA memory 0.03 GB
2026-10-06 15:42:55,111 INFO lasser.run: co-expression graph used (gears_static): {'num_nodes': 5045, 'num_edges': 6364, 'num_self_loops': 4699, 'degree_min': 0, 'degree_median': 0.0, 'degree_mean': 0.33002973240832506, 'degree_max': 20, 'degree_gini': 0.9324442777762897, 'isolated_genes': 4419}
2026-10-06 15:42:55,112 INFO lasser.run: training GEARS on cuda:0 (Tesla T4, 14.6 GB) for 20 epochs
2026-10-06 15:42:55,113 INFO lasser.utils: [gears_train] start


Start Training...
Epoch 1 Step 1 Train Loss: 0.5244
Epoch 1 Step 51 Train Loss: 0.6091
Epoch 1 Step 101 Train Loss: 0.5209
Epoch 1 Step 151 Train Loss: 0.5209
Epoch 1 Step 201 Train Loss: 0.4587
Epoch 1 Step 251 Train Loss: 0.5997
Epoch 1 Step 301 Train Loss: 0.4134
Epoch 1 Step 351 Train Loss: 0.5552
Epoch 1 Step 401 Train Loss: 0.5103
Epoch 1 Step 451 Train Loss: 0.5323
Epoch 1 Step 501 Train Loss: 0.4438
Epoch 1 Step 551 Train Loss: 0.6089
Epoch 1 Step 601 Train Loss: 0.4665
Epoch 1 Step 651 Train Loss: 0.5068
Epoch 1 Step 701 Train Loss: 0.4650
Epoch 1 Step 751 Train Loss: 0.4468
Epoch 1 Step 801 Train Loss: 0.4661
Epoch 1 Step 851 Train Loss: 0.4330
Epoch 1 Step 901 Train Loss: 0.5428
Epoch 1 Step 951 Train Loss: 0.4253
Epoch 1 Step 1001 Train Loss: 0.4217
Epoch 1 Step 1051 Train Loss: 0.4812
Epoch 1 Step 1101 Train Loss: 0.5036
Epoch 1 Step 1151 Train Loss: 0.4303
Epoch 1 Step 1201 Train Loss: 0.4306
Epoch 1 Step 1251 Train Loss: 0.4368
Epoch 1 Step 1301 Train Loss: 0.4825
Epoch 

2026-10-06 17:05:50,246 INFO lasser.utils: [gears_train] done in 4975.1s, peak CUDA memory 0.87 GB
2026-10-06 17:05:50,270 INFO lasser.utils: [evaluate] start
2026-10-06 17:08:50,740 INFO lasser.evaluation: test (eval.py): pearson_delta=0.5338 mse_top20_de=0.2105 nmse_top20_de=1.4284 over 107 perts
2026-10-06 17:08:50,741 INFO lasser.evaluation:   combo_seen0    pearson_delta=0.6349 mse_top20_de=0.1419 (9 perts)
2026-10-06 17:08:50,741 INFO lasser.evaluation:   combo_seen1    pearson_delta=0.5586 mse_top20_de=0.2077 (43 perts)
2026-10-06 17:08:50,742 INFO lasser.evaluation:   combo_seen2    pearson_delta=0.5626 mse_top20_de=0.2429 (19 perts)
2026-10-06 17:08:50,743 INFO lasser.evaluation:   unseen_single  pearson_delta=0.4636 mse_top20_de=0.2139 (36 perts)
2026-10-06 17:08:50,745 INFO lasser.utils: [evaluate] done in 180.5s, peak CUDA memory 1.15 GB
2026-10-06 17:08:50,765 INFO lasser.tracking: appended run f71d4a24 to /kaggle/working/runs/runs.csv


'/kaggle/working/runs/norman_seed1_base_20261006-154235'

In [7]:
# 6. Graph learning: GSR-refined co-expression graph (defaults from the plan, section 11).
gl = run_experiment(LasserConfig(**common, graph_learning=True, gsr=GSRConfig()), pert_data)
gl["run_dir"], gl["checks"]["h2"], gl["checks"]["refine"]

2026-10-06 17:08:50,840 INFO lasser.tracking: run 493c3ac6 -> /kaggle/working/runs/norman_seed1_gl_20261006-170850
2026-10-06 17:08:50,849 INFO lasser.run: device: cuda:0 (Tesla T4, 14.6 GB) (GEARS training and GSR graph learning)


Local copy of split is detected. Loading...
Simulation split test composition:
combo_seen0:9
combo_seen1:43
combo_seen2:19
unseen_single:36
Done!
Creating dataloaders....
Done!


here1
2026-10-06 17:08:50,864 INFO lasser.tracking: split hash f93e7d59ee239960 (copied ['norman_simulation_1_0.75_subgroup.pkl', 'norman_simulation_1_0.75.pkl'])
2026-10-06 17:08:50,867 INFO lasser.utils: [gsr_graph] start
2026-10-06 17:08:50,923 INFO lasser.graph: GSR graph learning on cuda:0 (Tesla T4, 14.6 GB)
2026-10-06 17:08:50,924 INFO lasser.utils: [gsr/data] start
2026-10-06 17:08:51,456 INFO lasser.data: training cells: 49849 (A0 subset 38824, H1 24957, H2 24892) over 138 training perturbations
2026-10-06 17:08:51,457 INFO lasser.utils: [gsr/data] done in 0.5s, peak CUDA memory 0.05 GB
2026-10-06 17:08:51,458 INFO lasser.utils: [gsr/views] start
2026-10-06 17:09:01,444 INFO lasser.data: metacells: 49849 cells -> 500 metacells
2026-10-06 17:09:02,773 INFO lasser.views: view E: dim 64, missing genes 234
2026-10-06 17:09:03,271 INFO lasser.views: view R: dim 32, missing genes 271
2026-10-06 17:09:03,669 INFO lasser.views: deepwalk epoch 1: loss 1.6828 over 1064200 pairs
2026-10-

Found local copy...


2026-10-06 17:09:32,830 INFO lasser.utils: [gears_init] done in 18.8s, peak CUDA memory 0.08 GB
2026-10-06 17:09:32,835 INFO lasser.run: co-expression graph used (gsr_refined): {'num_nodes': 5045, 'num_edges': 45192, 'num_self_loops': 4699, 'degree_min': 5, 'degree_median': 7.0, 'degree_mean': 8.026362735381566, 'degree_max': 25, 'degree_gini': 0.20116415036019025, 'isolated_genes': 0}
2026-10-06 17:09:32,924 INFO lasser.run: training GEARS on cuda:0 (Tesla T4, 14.6 GB) for 20 epochs
2026-10-06 17:09:32,925 INFO lasser.utils: [gears_train] start


Start Training...
Epoch 1 Step 1 Train Loss: 0.5244
Epoch 1 Step 51 Train Loss: 0.6062
Epoch 1 Step 101 Train Loss: 0.5209
Epoch 1 Step 151 Train Loss: 0.5244
Epoch 1 Step 201 Train Loss: 0.4638
Epoch 1 Step 251 Train Loss: 0.5955
Epoch 1 Step 301 Train Loss: 0.4118
Epoch 1 Step 351 Train Loss: 0.5500
Epoch 1 Step 401 Train Loss: 0.5074
Epoch 1 Step 451 Train Loss: 0.5306
Epoch 1 Step 501 Train Loss: 0.4490
Epoch 1 Step 551 Train Loss: 0.6113
Epoch 1 Step 601 Train Loss: 0.4674
Epoch 1 Step 651 Train Loss: 0.5061
Epoch 1 Step 701 Train Loss: 0.4643
Epoch 1 Step 751 Train Loss: 0.4464
Epoch 1 Step 801 Train Loss: 0.4670
Epoch 1 Step 851 Train Loss: 0.4354
Epoch 1 Step 901 Train Loss: 0.5447
Epoch 1 Step 951 Train Loss: 0.4254
Epoch 1 Step 1001 Train Loss: 0.4187
Epoch 1 Step 1051 Train Loss: 0.4877
Epoch 1 Step 1101 Train Loss: 0.5040
Epoch 1 Step 1151 Train Loss: 0.4317
Epoch 1 Step 1201 Train Loss: 0.4371
Epoch 1 Step 1251 Train Loss: 0.4386
Epoch 1 Step 1301 Train Loss: 0.4777
Epoch 

2026-10-06 18:32:37,753 INFO lasser.utils: [gears_train] done in 4984.8s, peak CUDA memory 0.91 GB
2026-10-06 18:32:37,775 INFO lasser.utils: [evaluate] start
2026-10-06 18:35:38,205 INFO lasser.evaluation: test (eval.py): pearson_delta=0.5745 mse_top20_de=0.1875 nmse_top20_de=1.0387 over 107 perts
2026-10-06 18:35:38,206 INFO lasser.evaluation:   combo_seen0    pearson_delta=0.6465 mse_top20_de=0.1467 (9 perts)
2026-10-06 18:35:38,206 INFO lasser.evaluation:   combo_seen1    pearson_delta=0.6005 mse_top20_de=0.1976 (43 perts)
2026-10-06 18:35:38,207 INFO lasser.evaluation:   combo_seen2    pearson_delta=0.6332 mse_top20_de=0.1849 (19 perts)
2026-10-06 18:35:38,208 INFO lasser.evaluation:   unseen_single  pearson_delta=0.4943 mse_top20_de=0.1871 (36 perts)
2026-10-06 18:35:38,210 INFO lasser.utils: [evaluate] done in 180.4s, peak CUDA memory 1.19 GB
2026-10-06 18:35:38,235 INFO lasser.tracking: appended run 493c3ac6 to /kaggle/working/runs/runs.csv


('/kaggle/working/runs/norman_seed1_gl_20261006-170850',
 {'threshold': 0.2,
  'hit_rate_added': 0.03598640859127045,
  'hit_rate_added_only': 0.03598640859127045,
  'hit_rate_topup': nan,
  'hit_rate_degree_matched_random': 0.0016736617544665933,
  'hit_rate_a0': 0.5806451439857483,
  'hit_rate_removed': 0.75,
  'passes': True},
 {'a0_edges': 6364,
  'removed': 20,
  'added_directed': 38848,
  'topup_directed': 0,
  'refined_edges': 45192,
  'new_weight_mean': 0.24045628309249878,
  'a0_weight_mean': 0.5615240931510925})

In [8]:
# 7. Results: the global run table, and any past run loaded back.
import pandas as pd
pd.set_option("display.max_columns", 200)
runs = pd.read_csv("/kaggle/working/runs/runs.csv")
cols = ["run_id", "seed", "graph_learning", "gsr_init_gene_emb", "graph_num_edges", "graph_isolated_genes",
        "test_pearson_delta", "test_mse_top20_de", "test_nmse_top20_de",
        "test_unseen_single_pearson_delta", "test_combo_seen0_pearson_delta",
        "test_combo_seen1_pearson_delta", "test_combo_seen2_pearson_delta", "run_dir"]
display(runs[[c for c in cols if c in runs.columns]])

r = load_run(gl["run_dir"])
display(r["epoch_log"])
r["test_metrics"]["paper"]["subgroups"]

,run_id,seed,graph_learning,gsr_init_gene_emb,graph_num_edges,graph_isolated_genes,test_pearson_delta,test_mse_top20_de,test_nmse_top20_de,test_unseen_single_pearson_delta,test_combo_seen0_pearson_delta,test_combo_seen1_pearson_delta,test_combo_seen2_pearson_delta,run_dir
0,f71d4a24,1,False,False,6364,4419,0.533766,0.210485,1.428422,0.463590,0.634930,0.558603,0.562602,/kaggle/working/runs/norman_seed1_base_2026100...
1,493c3ac6,1,True,False,45192,0,0.574455,0.187517,1.038668,0.494295,0.646518,0.600508,0.633239,/kaggle/working/runs/norman_seed1_gl_20261006-...


,epoch,train_mse,val_mse,train_mse_de,val_mse_de,train_loss_logged_mean,n_logged_steps,is_best
0,1,0.0032,0.0041,0.1239,0.2875,0.483009,32,True
1,2,0.0040,0.0048,0.1317,0.3456,0.496663,32,False
2,3,0.0024,0.0034,0.0868,0.2366,0.498409,32,True
3,4,0.0024,0.0037,0.0926,0.2610,0.502200,32,False
4,5,0.0023,0.0033,0.0742,0.2097,0.522313,32,True
5,6,0.0021,0.0032,0.0735,0.2126,0.503419,32,False
6,7,0.0022,0.0035,0.0715,0.2334,0.506322,32,False
7,8,0.0021,0.0033,0.0716,0.2269,0.533853,32,False
8,9,0.0022,0.0035,0.0680,0.2194,0.531909,32,False
9,10,0.0020,0.0033,0.0774,0.2231,0.503912,32,False


{'combo_seen0': {'n_perts': 9,
  'mse': 0.004927011206746101,
  'pearson': 0.9859106227311661,
  'mse_top20_de': 0.14665594427949852,
  'mse_top20_de_noperturb': 0.3686102908104658,
  'nmse_top20_de': 3.4552874007028884,
  'pearson_top20_de': 0.747353105451855,
  'pearson_delta': 0.6465183858281702,
  'pearson_delta_top20_de': 0.6548946607209559,
  'frac_opposite_top20_de': 0.5333333333333334,
  'frac_correct_direction_top20_de': 0.46666666666666656,
  'nmse_top20_de_ratio_of_means': 0.397861774170887},
 'combo_seen1': {'n_perts': 43,
  'mse': 0.00549168914399478,
  'pearson': 0.9845545444806643,
  'mse_top20_de': 0.19756975584815062,
  'mse_top20_de_noperturb': 0.48487508398762275,
  'nmse_top20_de': 0.9438417085966495,
  'pearson_top20_de': 0.8624252416685393,
  'pearson_delta': 0.6005083183312933,
  'pearson_delta_top20_de': 0.710915503746452,
  'frac_opposite_top20_de': 0.3581395348837209,
  'frac_correct_direction_top20_de': 0.6418604651162791,
  'nmse_top20_de_ratio_of_means': 0.